# Extend food dataset

Run setup first. Then run sync. Then test with a small `SELECTED_SLUGS` subset before full scrape.


In [1]:
%pip install -q pillow requests torch transformers playwright imagehash sentencepiece
!playwright install --with-deps chromium



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Installing dependencies...
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1).
fonts-liberation is already the newest version (1:1.07.4-11).
libatk-bridge2.0-0 is already the newest version (2.38.0-3).
libatk1.0-0 is already the newest version (2.36.0-3build1).
libatspi

In [2]:
import io
import random
import shutil
import urllib.parse
from collections import Counter
from pathlib import Path

import imagehash
import numpy as np
import requests
import torch
from PIL import Image, ImageOps
from playwright.async_api import async_playwright
from transformers import AutoModel, AutoProcessor

DATA_ROOT = Path('/tf/data/food-extended/images')
CLASSES_PATH = Path('/tf/ML/final/classes.txt')
LABELS_PATH = Path('/tf/ML/final/labels.txt')
TARGET_IMAGES_PER_CLASS = 100
MAX_EDGE = 512
MIN_SOURCE_EDGE = 256
MAX_URLS_PER_CLASS = 250
SCROLL_ROUNDS = 10
PHASH_THRESHOLD = 5
MIN_TARGET_SCORE = 0.60
MIN_SCORE_MARGIN = 0.05
NEGATIVE_LABEL_COUNT = 7
SIGLIP_MODEL_ID = 'google/siglip-base-patch16-224'
DRY_RUN_DELETE = False
HEADLESS = True
SELECTED_SLUGS = None
VERBOSE_LOGGING = True
USER_AGENT = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
VALID_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def read_lines(path):
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def log_event(*parts):
    if VERBOSE_LOGGING:
        print(*parts)

allowed_slugs = read_lines(CLASSES_PATH)
labels = read_lines(LABELS_PATH)
if len(allowed_slugs) != len(labels):
    raise ValueError(f'classes.txt has {len(allowed_slugs)} rows but labels.txt has {len(labels)} rows')

slug_to_label = dict(zip(allowed_slugs, labels))
print(f'Loaded {len(allowed_slugs)} classes and {len(labels)} labels')
print('Sample:', allowed_slugs[:5], '->', labels[:5])


Loaded 904 classes and 904 labels
Sample: ['acai_berry', 'acai_bowl', 'acerola', 'ackee', 'adobo'] -> ['Acai Berry', 'Acai Bowl', 'Acerola', 'Ackee', 'Adobo']


In [3]:
def list_image_files(class_dir):
    return sorted([path for path in class_dir.iterdir() if path.is_file() and path.suffix.lower() in VALID_SUFFIXES])

def sync_dataset_folders(data_root, valid_slugs, dry_run=False):
    if not data_root.exists():
        raise FileNotFoundError(f'{data_root} does not exist')

    valid_slug_set = set(valid_slugs)
    existing_dirs = sorted([path for path in data_root.iterdir() if path.is_dir()])
    deleted = []

    for path in existing_dirs:
        if path.name not in valid_slug_set:
            deleted.append(path.name)
            if not dry_run:
                shutil.rmtree(path)

    created = []
    for slug in valid_slugs:
        class_dir = data_root / slug
        if not class_dir.exists():
            created.append(slug)
            if not dry_run:
                class_dir.mkdir(parents=True, exist_ok=True)

    print(f'Deleted {len(deleted)} folders')
    if deleted:
        print(deleted)
    print(f'Created {len(created)} missing folders')
    return deleted, created

deleted_folders, created_folders = sync_dataset_folders(DATA_ROOT, allowed_slugs, dry_run=DRY_RUN_DELETE)


Deleted 0 folders
Created 0 missing folders


In [5]:
def resize_to_max_edge(image, max_edge=512):
    image = ImageOps.exif_transpose(image).convert('RGB')
    width, height = image.size
    largest_edge = max(width, height)
    if largest_edge <= max_edge:
        return image
    scale = max_edge / float(largest_edge)
    new_size = (max(1, round(width * scale)), max(1, round(height * scale)))
    return image.resize(new_size, Image.Resampling.LANCZOS)

def load_existing_hashes(class_dir):
    hashes = []
    for path in list_image_files(class_dir):
        try:
            with Image.open(path) as image:
                hashes.append(imagehash.phash(resize_to_max_edge(image, MAX_EDGE)))
        except Exception:
            continue
    return hashes

def is_near_duplicate(image, existing_hashes, threshold=PHASH_THRESHOLD):
    candidate_hash = imagehash.phash(image)
    for existing_hash in existing_hashes:
        if candidate_hash - existing_hash <= threshold:
            return True, candidate_hash
    return False, candidate_hash

def save_image_if_new(image, class_dir, existing_hashes):
    prepared = resize_to_max_edge(image, MAX_EDGE)
    duplicate, candidate_hash = is_near_duplicate(prepared, existing_hashes)
    if duplicate:
        return None, 'duplicate_phash'

    buffer = io.BytesIO()
    prepared.save(buffer, format='JPEG', quality=95, optimize=True)
    image_bytes = buffer.getvalue()
    filename = f"{sum(image_bytes) % 10**12:012d}_{candidate_hash}.jpg"
    output_path = class_dir / filename
    while output_path.exists():
        filename = f"{random.randint(0, 10**12 - 1):012d}_{candidate_hash}.jpg"
        output_path = class_dir / filename

    output_path.write_bytes(image_bytes)
    existing_hashes.append(candidate_hash)
    return output_path, 'saved'


In [6]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

try:
    import sentencepiece  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "SentencePiece missing. Run install cell, then restart kernel, then rerun from top."
    ) from exc

processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID)
model = AutoModel.from_pretrained(SIGLIP_MODEL_ID, torch_dtype=DTYPE)
model.to(DEVICE)
model.eval()
print(f'SigLIP loaded on {DEVICE}')

def build_candidate_labels(target_slug, target_label, all_slug_to_label, negative_count=NEGATIVE_LABEL_COUNT):
    other_slugs = [slug for slug in all_slug_to_label if slug != target_slug]
    rng = random.Random(target_slug)
    sampled_slugs = rng.sample(other_slugs, k=min(negative_count, len(other_slugs)))
    sampled_labels = [all_slug_to_label[slug] for slug in sampled_slugs]
    return [target_label] + sampled_labels

def score_image_against_labels(image, candidate_labels):
    texts = [f'This is a photo of {label}.' for label in candidate_labels]
    inputs = processor(text=texts, images=image, padding='max_length', return_tensors='pt')
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    return torch.sigmoid(outputs.logits_per_image)[0].detach().float().cpu().numpy()

def image_matches_label(image, target_slug, target_label, all_slug_to_label):
    candidate_labels = build_candidate_labels(target_slug, target_label, all_slug_to_label)
    probs = score_image_against_labels(image, candidate_labels)
    best_idx = int(np.argmax(probs))
    target_score = float(probs[0])
    strongest_other = float(max(probs[1:])) if len(probs) > 1 else 0.0
    accepted = best_idx == 0 and target_score >= MIN_TARGET_SCORE and (target_score - strongest_other) >= MIN_SCORE_MARGIN
    return accepted, {
        'target_score': round(target_score, 4),
        'best_label': candidate_labels[best_idx],
        'best_score': round(float(probs[best_idx]), 4),
        'strongest_other': round(strongest_other, 4),
        'candidate_labels': candidate_labels
    }


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP loaded on cuda


In [7]:
session = requests.Session()
session.headers.update({'User-Agent': USER_AGENT})

async def maybe_accept_google_consent(page):
    button_labels = ['Accept all', 'I agree', 'Accept', 'Alle akzeptieren', 'Tout accepter']
    for label in button_labels:
        try:
            await page.get_by_role('button', name=label).click(timeout=1500)
            await page.wait_for_timeout(1000)
            log_event('  consent accepted:', label)
            return True
        except Exception:
            continue
    return False

async def collect_http_image_urls(page):
    urls = await page.locator('img').evaluate_all(
        """(nodes) => nodes.map((node) => node.currentSrc || node.src || node.getAttribute('data-src') || '').filter((src) => src.startsWith('http'))"""
    )
    cleaned = []
    seen = set()
    for url in urls:
        lowered = url.lower()
        if 'gstatic.com' in lowered or 'google.com' in lowered:
            continue
        if url not in seen:
            seen.add(url)
            cleaned.append(url)
    return cleaned

async def search_google_image_urls(page, query, max_urls=MAX_URLS_PER_CLASS, scroll_rounds=SCROLL_ROUNDS):
    search_url = 'https://www.google.com/search?tbm=isch&q=' + urllib.parse.quote_plus(query)
    await page.goto(search_url, wait_until='domcontentloaded', timeout=60000)
    await page.wait_for_timeout(1500)
    log_event('  search url:', search_url)
    log_event('  page title:', await page.title())
    await maybe_accept_google_consent(page)

    collected = []
    seen = set()
    for round_idx in range(scroll_rounds):
        thumbnails = page.locator('img')
        thumb_count = min(await thumbnails.count(), 50)
        log_event(f'  round {round_idx + 1}/{scroll_rounds}: thumbnails={thumb_count} collected={len(collected)}')
        for idx in range(thumb_count):
            try:
                await thumbnails.nth(idx).click(timeout=1200)
                await page.wait_for_timeout(500)
            except Exception:
                pass

            for url in await collect_http_image_urls(page):
                if url not in seen:
                    seen.add(url)
                    collected.append(url)
                    if len(collected) >= max_urls:
                        log_event('  collected max urls:', len(collected))
                        return collected

        await page.mouse.wheel(0, 5000)
        await page.wait_for_timeout(1200)

    log_event('  collected urls total:', len(collected))
    return collected

def download_image(url):
    response = session.get(url, timeout=20)
    response.raise_for_status()
    content_type = response.headers.get('content-type', '').lower()
    if 'image' not in content_type:
        raise ValueError(f'URL did not return image content: {content_type}')
    image = Image.open(io.BytesIO(response.content))
    image.load()
    return image

def process_candidate_url(url, class_dir, target_slug, target_label, existing_hashes):
    try:
        image = download_image(url)
    except Exception as exc:
        return None, 'download_error', {'url': url, 'error': repr(exc)}

    if max(image.size) < MIN_SOURCE_EDGE:
        return None, 'too_small', {'url': url, 'size': image.size}

    resized = resize_to_max_edge(image, MAX_EDGE)
    accepted, score_info = image_matches_label(resized, target_slug, target_label, slug_to_label)
    score_info['url'] = url
    score_info['size'] = resized.size
    if not accepted:
        return None, 'siglip_reject', score_info

    output_path, save_reason = save_image_if_new(resized, class_dir, existing_hashes)
    score_info['saved_to'] = str(output_path) if output_path else None
    return output_path, save_reason, score_info


In [8]:
async def extend_one_class(page, slug, label, target_count=TARGET_IMAGES_PER_CLASS):
    class_dir = DATA_ROOT / slug
    class_dir.mkdir(parents=True, exist_ok=True)
    existing_files = list_image_files(class_dir)
    existing_hashes = load_existing_hashes(class_dir)
    final_count = len(existing_files)
    stats = Counter()
    stats['existing'] = final_count

    if final_count >= target_count:
        log_event(f'  skip: already at target with {final_count} images')
        return {'slug': slug, 'label': label, 'existing': final_count, 'added': 0, 'final': final_count, 'rejections': {}}

    query = f'{label} food'
    candidate_urls = await search_google_image_urls(page, query, max_urls=MAX_URLS_PER_CLASS, scroll_rounds=SCROLL_ROUNDS)
    seen_urls = set()
    log_event(f'  candidate url count: {len(candidate_urls)}')

    if not candidate_urls:
        log_event('  no candidate URLs found. Likely Google blocked results or selectors did not match page structure.')

    for idx, url in enumerate(candidate_urls, start=1):
        if final_count >= target_count:
            break
        if url in seen_urls:
            continue
        seen_urls.add(url)

        log_event(f'  candidate {idx}/{len(candidate_urls)}: {url}')
        output_path, result, details = process_candidate_url(url, class_dir, slug, label, existing_hashes)
        stats[result] += 1

        if result == 'saved':
            final_count += 1
            log_event(
                f"    saved -> {output_path.name}",
                f"score={details['target_score']}",
                f"best={details['best_label']}",
                f"other={details['strongest_other']}"
            )
        elif result == 'siglip_reject':
            log_event(
                '    rejected by siglip',
                f"score={details['target_score']}",
                f"best={details['best_label']}",
                f"best_score={details['best_score']}",
                f"other={details['strongest_other']}"
            )
        elif result == 'too_small':
            log_event('    rejected too_small', details['size'])
        elif result == 'download_error':
            log_event('    download_error', details['error'])
        elif result == 'duplicate_phash':
            log_event('    rejected duplicate_phash')
        else:
            log_event('    result', result, details)

    log_event('  class summary:', dict(stats))
    return {
        'slug': slug,
        'label': label,
        'existing': stats['existing'],
        'added': stats['saved'],
        'final': final_count,
        'rejections': dict(stats)
    }

async def extend_dataset(selected_slugs=None, headless=HEADLESS):
    run_slugs = selected_slugs or allowed_slugs
    results = []

    async with async_playwright() as playwright:
        try:
            browser = await playwright.chromium.launch(headless=headless)
        except Exception as exc:
            raise RuntimeError(
                'Chromium launch failed. Run install cell again so Playwright installs Linux browser deps, then restart kernel and retry.'
            ) from exc

        context = await browser.new_context(user_agent=USER_AGENT, locale='en-US', viewport={'width': 1600, 'height': 1200})
        page = await context.new_page()

        for idx, slug in enumerate(run_slugs, start=1):
            label = slug_to_label[slug]
            print(f'[{idx}/{len(run_slugs)}] {slug} -> {label}')
            result = await extend_one_class(page, slug, label)
            results.append(result)
            print(f"  existing={result['existing']} added={result['added']} final={result['final']}")
            print(f"  rejection summary={result['rejections']}")

        await context.close()
        await browser.close()

    return results


In [9]:
results = await extend_dataset(selected_slugs=SELECTED_SLUGS, headless=HEADLESS)
results[:3]


[1/904] acai_berry -> Acai Berry
  search url: https://www.google.com/search?tbm=isch&q=Acai+Berry+food
  page title: https://www.google.com/search?q=Acai+Berry+food&udm=2&sei=0Ff9abeBNOSehbIPtdbmoQI
  round 1/10: thumbnails=0 collected=0
  round 2/10: thumbnails=0 collected=0
  round 3/10: thumbnails=0 collected=0
  round 4/10: thumbnails=0 collected=0
  round 5/10: thumbnails=0 collected=0
  round 6/10: thumbnails=0 collected=0
  round 7/10: thumbnails=0 collected=0
  round 8/10: thumbnails=0 collected=0
  round 9/10: thumbnails=0 collected=0
  round 10/10: thumbnails=0 collected=0
  collected urls total: 0
  candidate url count: 0
  no candidate URLs found. Likely Google blocked results or selectors did not match page structure.
  class summary: {'existing': 0}
  existing=0 added=0 final=0
  rejection summary={'existing': 0}
[2/904] acai_bowl -> Acai Bowl
  search url: https://www.google.com/search?tbm=isch&q=Acai+Bowl+food
  page title: https://www.google.com/search?q=Acai+Bowl+foo

/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:935: UserWarning: Truncated File Read
  warnings.warn(str(msg))


  skip: already at target with 1000 images
  existing=1000 added=0 final=1000
  rejection summary={}
[21/904] apricot -> Apricot
  round 7/10: thumbnails=0 collected=0
  round 8/10: thumbnails=0 collected=0
  round 9/10: thumbnails=0 collected=0
  round 10/10: thumbnails=0 collected=0
  collected urls total: 0
  candidate url count: 0
  no candidate URLs found. Likely Google blocked results or selectors did not match page structure.
  class summary: {'existing': 0}
  existing=0 added=0 final=0
  rejection summary={'existing': 0}
[27/904] avgolemono -> Avgolemono
  search url: https://www.google.com/search?tbm=isch&q=Avgolemono+food
  page title: https://www.google.com/search?q=Avgolemono+food&udm=2&sei=LFr9aaPuJrbNhbIPhbqIwAY
  round 1/10: thumbnails=0 collected=0
  round 2/10: thumbnails=0 collected=0
  round 3/10: thumbnails=0 collected=0
  round 4/10: thumbnails=0 collected=0
  round 5/10: thumbnails=0 collected=0
  round 6/10: thumbnails=0 collected=0
  round 7/10: thumbnails=0 col

Error: Locator.evaluate_all: Execution context was destroyed, most likely because of a navigation

In [ ]:
def summarize_counts(selected_slugs=None):
    run_slugs = selected_slugs or allowed_slugs
    summary = []
    for slug in run_slugs:
        class_dir = DATA_ROOT / slug
        count = len(list_image_files(class_dir)) if class_dir.exists() else 0
        summary.append({'slug': slug, 'label': slug_to_label[slug], 'count': count})
    return summary

summary = summarize_counts(selected_slugs=SELECTED_SLUGS)
below_target = [row for row in summary if row['count'] < TARGET_IMAGES_PER_CLASS]
print(f'Classes below target: {len(below_target)}')
print('First 10 below target:', below_target[:10])
print('Deleted folders:', deleted_folders)
